In [ ]:
import pandas as pd
import numpy as np
import re

from collections import Counter, defaultdict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
DEV_PATH  = "../data/raw/development.csv"
EVAL_PATH = "../data/raw/evaluation.csv"
SUB_PATH  = "submission_tristage.csv"
MIN_RULE_SUPPORT = 34
MIN_RULE_PURITY  = 0.926328564964768

WORD_NG_MAX = 2
CHAR_NG_MAX = 5
MIN_DF      = 2
MAX_DF      = 0.8782583211530898
C_VALUE     = 0.64491922705094

df_dev  = pd.read_csv(DEV_PATH)
df_eval = pd.read_csv(EVAL_PATH)

for df in (df_dev, df_eval):
	for c in ["article", "title", "source", "timestamp"]:
		df[c] = df[c].fillna("").astype(str)
def build_text(df):
	return (df["title"] + " " + df["article"]).str.lower()

def add_numeric(df):
	df["n_tokens"]    = df["article"].str.split().str.len()
	df["title_len"]   = df["title"].str.len()
	df["article_len"] = df["article"].str.len()
	df["title_ratio"] = df["title_len"] / (df["article_len"] + 1)
	return df

df_dev["text"]  = build_text(df_dev)
df_eval["text"] = build_text(df_eval)

df_dev  = add_numeric(df_dev)
df_eval = add_numeric(df_eval)

NUM_COLS = ["n_tokens", "title_len", "article_len", "title_ratio"]

for df in (df_dev, df_eval):
	df[NUM_COLS] = df[NUM_COLS].replace([np.inf, -np.inf], 0).fillna(0)
def build_first_ts_label(df):
	df = df.copy()
	df = df[df["timestamp"] != "0000-00-00 00:00:00"]
	df["ts"] = pd.to_datetime(df["timestamp"], errors="coerce")

	dup = df[df.duplicated("article", keep=False)]
	first = (
		dup.sort_values("ts")
		   .groupby("article")
		   .first()["label"]
	)
	return first.to_dict()

FIRST_TS_LABEL = build_first_ts_label(df_dev)

def apply_stage1(df):
	pred = pd.Series(index=df.index, dtype="float")
	mask = df["article"].isin(FIRST_TS_LABEL)
	pred[mask] = df.loc[mask, "article"].map(FIRST_TS_LABEL)
	return pred, mask
def tokenize(text):
	return re.findall(r"[a-z0-9_:/\.]+", text.lower())

def mine_rules(texts, labels):
	counts = defaultdict(lambda: Counter())

	for t, y in zip(texts, labels):
		for tok in set(tokenize(t)):
			counts[tok][int(y)] += 1

	rules = {}
	meta  = {}

	for tok, c in counts.items():
		total = sum(c.values())
		if total < MIN_RULE_SUPPORT:
			continue

		lbl, freq = c.most_common(1)[0]
		purity = freq / total

		if purity >= MIN_RULE_PURITY:
			rules[tok] = lbl
			meta[tok]  = (purity, total)

	return rules, meta

def apply_rules(texts, rules, meta):
	out = np.full(len(texts), -1)
	for i, t in enumerate(texts):
		hits = [tok for tok in set(tokenize(t)) if tok in rules]
		if not hits:
			continue
		hits.sort(key=lambda x: (meta[x][0], meta[x][1]), reverse=True)
		out[i] = rules[hits[0]]
	return out
RULES, RULE_META = mine_rules(df_dev["article"], df_dev["label"])
def make_model():
	pre = ColumnTransformer(
		[
			("src", OneHotEncoder(handle_unknown="ignore"), ["source"]),
			("w", TfidfVectorizer(
				ngram_range=(1, WORD_NG_MAX),
				min_df=MIN_DF,
				max_df=MAX_DF,
				sublinear_tf=True,
				max_features=250_000
			), "text"),
			("c", TfidfVectorizer(
				analyzer="char_wb",
				ngram_range=(3, CHAR_NG_MAX),
				min_df=MIN_DF,
				max_df=MAX_DF,
				sublinear_tf=True,
				max_features=300_000
			), "text"),
			("num", StandardScaler(), NUM_COLS),
		],
		n_jobs=-1
	)

	clf = LogisticRegression(
		C=C_VALUE,
		class_weight="balanced",
		max_iter=2000,
		n_jobs=-1
	)

	return Pipeline([("pre", pre), ("clf", clf)])
model = make_model()
model.fit(df_dev[["source","text"] + NUM_COLS], df_dev["label"])
# Stage 1
s1_pred, s1_mask = apply_stage1(df_eval)

# Stage 2
s2_pred = apply_rules(df_eval["article"], RULES, RULE_META)
s2_mask = (s1_mask == False) & (s2_pred != -1)

# Stage 3
s3_mask = ~(s1_mask | s2_mask)
s3_pred = model.predict(df_eval.loc[s3_mask, ["source","text"] + NUM_COLS])

# Merge
final_pred = np.empty(len(df_eval))
final_pred[s1_mask] = s1_pred[s1_mask]
final_pred[s2_mask] = s2_pred[s2_mask]
final_pred[s3_mask] = s3_pred
submission = pd.DataFrame({
	"Id": df_eval["Id"].astype(int),
	"Predicted": final_pred.astype(int)
})

submission.to_csv(SUB_PATH, index=False)
print("Saved:", SUB_PATH)


Saved: submission_tristage.csv


: 